In [1]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,586 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently

In [3]:
!sudo apt update && sudo apt install pciutils lshw

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]3m
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InReleasem
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease  
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,850 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]

GPU Enabled

In [2]:
!nvidia-smi

Wed Jul 29 19:36:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Running Ollama Server

Running on a cell leads to getting stuck in that cell, so run it as background process.

In [4]:
!ollama serve

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIDuNySTSoPGgFt8GvP9smk82kmtUTAmSBqkYb/z9Vhqo

time=2026-07-29T19:36:53.506Z level=INFO source=routes.go:1947 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost h

In [ ]:
#1. Kill any existing instances
!pkill ollama

# 2. Set environment variables -- OLLAMA_HOST binds the server to all network interfaces
# (0.0.0.0), not just localhost, in case you need to reach it from outside the container
# (e.g. a tunnel); OLLAMA_NUM_PARALLEL raises how many requests Ollama processes at once --
# was 1 (server default) when this cell first ran, so every request queued regardless of
# how many the client sent. 80GB VRAM on this A100 has plenty of room for several parallel
# 40960-context slots of a 7.6GB model, unlike the Mac's earlier concurrency benchmark.
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_NUM_PARALLEL'] = '8'

# 3. Start the server in the background
import subprocess
import time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE) # What is subprocess.PIPE?

# 4. Wait for it to warm up
time.sleep(10)
print("Server should be ready.")

Testing if Ollama is running

In [6]:
!curl http://localhost:11434 

Ollama is running

Install Ollama Python SDK

It makes using Ollama easy. We can access Ollama programmatically.

In [7]:
!pip install ollama

Download Gemma4:e2b Model: Run thos on terminal

/content 

ollama pull gemma4:e2b

Checking Model List In Ollama

In [8]:
!ollama list

NAME    ID    SIZE    MODIFIED 


Testing Gemma4 on Text Input

In [ ]:
import ollama
response = ollama.chat(model='gemma4:e2b', messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
])
print(response)

## ODER

In [ ]:
 !pip install -U transformers # What is -U for?

Local Inference on GPU
Model page: https://huggingface.co/google/gemma-4-12B-it

⚠️ If the generated code snippets do not work, please open an issue on either the model repo and/or on huggingface.js 🙏

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-12B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-12B-it", device_map="auto")

## Full-corpus run: gemma4:12b, Bavaria only

Scope: just Bavaria (`by`, AfD entry 2018-11-05), not the full ±1yr window across all
states -- 72,089 classifiable paragraphs vs. 1,144,158. Reason: Colab Pro has a **hard 24h
session cap** regardless of activity, and the full corpus would take an estimated ~56 days
at gemma4's local Mac rate -- Bavaria alone is ~3.5 days, i.e. only a handful of reconnects
instead of ~55.

`score_with_model.py --full-corpus` checkpoints every row to disk as it's scored and resumes
automatically from wherever it left off -- so when this session dies at the 24h mark, just
reconnect and re-run the last cell below. Uses the v2 window+flag prompt (see
`impoliteness_lib.py`'s `PROMPT_VERSION`).

Uses `gemma4:12b` -- NOT `gemma4:e2b` (that's the small edge/on-device variant, not the model
used in the interrater comparison).

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DATA_ROOT'] = "/content/drive/MyDrive/my_projects/M.A. Parliament/Code and Data/data"

Mounted at /content/drive


In [11]:
# Repo is public -- plain clone, no auth needed.
import os

if not os.path.isdir('/content/Incivility-in-Plenary-de'):
    !git clone https://github.com/AnLeWe/Incivility-in-Plenary-de.git /content/Incivility-in-Plenary-de
else:
    !cd /content/Incivility-in-Plenary-de && git pull

%cd /content/Incivility-in-Plenary-de
!pip install -q -r requirements.txt

Cloning into '/content/Incivility-in-Plenary-de'...
remote: Enumerating objects: 301, done.
remote: Counting objects: 100% (301/301), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 301 (delta 136), reused 297 (delta 132), pack-reused 0 (from 0)
Receiving objects: 100% (301/301), 1.43 MiB | 25.26 MiB/s, done.
Resolving deltas: 100% (136/136), done.
/content/Incivility-in-Plenary-de
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.6/349.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 152.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 148.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

Pull the model -- run in the terminal (Runtime → "Open terminal" or the `!` cell below), not
inline, same reason as before (progress bar doesn't render well in a notebook cell):

```
ollama pull gemma4:12b
```

In [12]:
!ollama pull gemma4:12b
!ollama list


NAME          ID              SIZE      MODIFIED               
gemma4:12b    4eb23ef187e2    7.6 GB    Less than a second ago    


## Concurrency benchmark (this GPU specifically)

Re-run the cell above (kill + restart `ollama serve`) first, so `OLLAMA_NUM_PARALLEL=8` is
actually in effect -- it printed `OLLAMA_NUM_PARALLEL:1` in the very first `ollama serve`
log further up, from before this was set.

Times real Bavaria rows (not toy prompts) at concurrency 1/2/4/8/16 with `gemma4:12b`, same
method as the earlier Mac benchmark. That one plateaued at concurrency=2 with only ~20%
gain -- compute-bound on that GPU. This A100 has a very different compute/VRAM ratio, so
don't assume the same ceiling; measure it here instead. Pick whatever concurrency actually
helps and set it in the `CONCURRENCY` variable in the next cell.

In [ ]:
import sys, time
sys.path.insert(0, '/content/Incivility-in-Plenary-de/measurement')
from concurrent.futures import ThreadPoolExecutor

import ollama
from impoliteness_lib import build_classification_pool, build_prompt, PROMPT_VERSION

pool = build_classification_pool(os.environ['DATA_ROOT'], verbose=False)
sample_texts = (
    pool.dedup[pool.dedup['state'] == 'by']['text_to_classify']
    .dropna().astype(str).head(16).tolist()
)

def call_one(text):
    t0 = time.time()
    ollama.chat(
        model='gemma4:12b',
        messages=build_prompt(text, prompt_version=PROMPT_VERSION),
        format='json', think=False,
        options={'temperature': 0, 'seed': 20260723, 'num_ctx': 40960},
    )
    return time.time() - t0

for concurrency in (1, 2, 4, 8, 16):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        list(ex.map(call_one, sample_texts))
    wall = time.time() - t0
    print(f"concurrency={concurrency}: {len(sample_texts)} items in {wall:.1f}s "
          f"({wall/len(sample_texts):.2f} sec/item wall-clock, "
          f"{len(sample_texts)/wall:.2f} items/sec throughput)")

In [ ]:
# Set this from the benchmark above -- whichever concurrency actually gave a real
# speedup (not just the highest number tried). 1 = sequential, same as before.
CONCURRENCY = 1

## Four separate variant runs, one cell each

Same idea as the local gemma4 ablation (n=2000 sample), but here on the full Bavaria corpus
(72,089 rows) instead of the fixed sample -- `baseline` is the actual v1 prompt (matches the
already-run n=2000 four-model comparison), `window`/`flag` isolate each half of v2's context,
`window_flag` is the full v2 design. Each is its own cell/output file so you can watch each
one separately rather than one chained loop.

Still run the four **cells** one after another, not simultaneously in separate tabs -- one
model instance, one GPU, running two cells at once would just contend for it. Within a single
cell, `--concurrency {CONCURRENCY}` (set from the benchmark above) sends that many requests to
Ollama at once, which is a different thing and is what this benchmark was for. Each cell is
independently checkpointed/resumable across 24h reconnects.

Rough time budget: ~3.5 days/variant, ~14 days for all four back to back, at `--concurrency 1`
and based on the local Mac gemma4 rate (~4.2s/item) -- both of those should be pessimistic
here: this session runs on an A100-SXM4-80GB (`nvidia-smi` above), not a T4, and the benchmark
above may justify a higher CONCURRENCY. Watch each cell's progress printout (every 25 items)
for the actual rate.

In [ ]:
# baseline -- actual v1 prompt, no context/flag at all (matches the n=2000 four-model comparison)
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant baseline --state by --platform-label colab_a100 --concurrency {CONCURRENCY}

In [ ]:
# window only -- +-1 paragraph context, ordnungsruf_follows suppressed
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant window --state by --platform-label colab_a100 --concurrency {CONCURRENCY}

In [ ]:
# flag only -- ordnungsruf_follows hint, no +-1 window
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant flag --state by --platform-label colab_a100 --concurrency {CONCURRENCY}

In [ ]:
# Run this cell to start, and re-run it (after reconnecting) whenever the session dies --
# it resumes from wherever impoliteness_full_by_gemma4-12b_window_flag.csv already has rows.
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant window_flag --state by --platform-label colab_a100 --concurrency {CONCURRENCY}